<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/10_deploy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10 · Deploy it

Everything so far has run inside a notebook. Studio served an agent that lived in your kernel,
and when the kernel died so did the agent.

This lesson turns one into a **deployment**: a real project on disk, a `langgraph.json` that
describes it, and a running service you call over the network. The agent you deploy uses the
pieces from lessons 01–08 — tools, a filesystem, a store, `AGENTS.md`, a skill, and middleware.

**New in this lesson:** `langgraph.json`, the `langgraph` CLI, LangSmith Deployments, `RemoteGraph`

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq --progress-bar off \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0" \
  "langgraph-cli[inmem]~=0.4.31" \
  "git+https://github.com/langchain-samples/langsmith-studio-nb.git"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-10-deploy"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

---

## 1. A project, not a notebook cell

A deployment needs a **directory**, because the platform has to build an image from it. Same
agent you have been writing, laid out as files:

```
support_agent/
├── langgraph.json          the deployment manifest
├── pyproject.toml          dependencies for the image
├── agent.py                builds the graph, exposes `agent`
├── data/
│   ├── orders.jsonl        our stand-in for a database
│   └── tickets.jsonl
├── memory/AGENTS.md        lesson 07 — house style, editable without a deploy
└── skills/
    └── refund-decision/
        └── SKILL.md        lesson 08 — a procedure loaded on demand
```

The `.jsonl` files are deliberate: a real deployment reads from somewhere, and a file you can
`cat` beats a mock you cannot inspect.

In [ ]:
!mkdir -p support_agent/data support_agent/memory support_agent/skills/refund-decision

In [ ]:
%%writefile support_agent/data/orders.jsonl
{"id": "1042", "customer": "avery@example.com", "item": "Standing desk", "status": "delivered", "days_ago": 3, "price": 429.00}
{"id": "1043", "customer": "jordan@example.com", "item": "Desk lamp", "status": "delivered", "days_ago": 45, "price": 39.00}
{"id": "1044", "customer": "avery@example.com", "item": "Monitor arm", "status": "in_transit", "days_ago": 1, "price": 89.00}
{"id": "1045", "customer": "sam@example.com", "item": "Office chair", "status": "delivered", "days_ago": 10, "price": 249.00}
{"id": "1046", "customer": "riley@example.com", "item": "Keyboard tray", "status": "cancelled", "days_ago": 7, "price": 59.00}
{"id": "1047", "customer": "sam@example.com", "item": "Laptop stand", "status": "delivered", "days_ago": 62, "price": 45.00}

In [ ]:
%%writefile support_agent/data/tickets.jsonl
{"id": "T-1", "order_id": "1042", "text": "Desk arrived with a cracked leg. Photos attached."}
{"id": "T-2", "order_id": "1043", "text": "Lamp stopped working. Bought it over a month ago."}
{"id": "T-3", "order_id": "1044", "text": "Where is my monitor arm? Ordered yesterday."}
{"id": "T-4", "order_id": "1045", "text": "Chair is fine but I ordered the wrong colour. Can I swap?"}
{"id": "T-5", "order_id": "1046", "text": "I cancelled this but was still charged."}
{"id": "T-6", "order_id": "1047", "text": "Laptop stand wobbles. Had it two months."}

In [ ]:
%%writefile support_agent/memory/AGENTS.md
# House style

- Address the customer by first name only.
- Never promise a delivery date. Say "typically 3-5 business days".
- Quote the exact policy line you relied on.
- British spelling.

In [ ]:
%%writefile support_agent/skills/refund-decision/SKILL.md
---
name: refund-decision
description: Decide what remedy a customer is entitled to. Use whenever a refund, replacement, exchange, or repair is being considered.
---

# Refund decision

## Procedure
1. Look up the order and note `days_ago` (days since delivery).
2. Classify the complaint: damaged-on-arrival, faulty, wrong-item, or cancelled-but-charged.
3. Apply the policy table below literally. Do not infer beyond it.
4. State the remedy, then quote the policy line you used.

## Policy
| Situation | Remedy |
|---|---|
| Damaged on arrival | Full refund or replacement, no time limit. Photos required. |
| Faulty within 30 days of delivery | Full refund or replacement. |
| Faulty after 30 days | Repair only. No refund, no replacement. |
| Wrong item ordered by customer | Exchange within 14 days. 10% restocking fee. |
| Cancelled but charged | Refund within 5 business days. Escalate. |

## Rules
- Refunds above $200 require human approval. Say so explicitly.
- Never offer a remedy the table does not permit.

---

## 2. The agent, as a module

Nothing here is new — it is lessons 01–08 in one file. The only structural difference is that a
deployment imports a module and looks for a variable, so the graph has to exist at import time.

In [ ]:
%%writefile support_agent/agent.py
"""Support agent, assembled from the pieces of lessons 01-08."""

from __future__ import annotations

import json
import pathlib

from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, FilesystemBackend, StateBackend, StoreBackend
from deepagents.middleware.filesystem import FilesystemPermission
from langchain.agents.middleware import (
    PIIMiddleware,
    SummarizationMiddleware,
    ToolRetryMiddleware,
)
from langchain_core.tools import tool

MODEL = "langsmith:openai/gpt-5.6-luna"
ROOT = pathlib.Path(__file__).parent


def _load(name: str) -> list[dict]:
    """Read one of the .jsonl 'tables' that stand in for a database."""
    path = ROOT / "data" / f"{name}.jsonl"
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


ORDERS = _load("orders")
TICKETS = _load("tickets")


def _memories_namespace(rt) -> tuple[str, ...]:
    """Scope /memories/ to whoever called the deployment."""
    info = rt.server_info
    identity = info.user.identity if info and info.user else "local"
    return ("memories", identity)


@tool
def lookup_order(order_id: str) -> str:
    """Look up a single order by its 4-digit ID.

    Returns item, status, days since delivery, price, and customer email.
    Use this before answering any question about a specific order.
    """
    for order in ORDERS:
        if order["id"] == order_id:
            return (
                f"Order {order['id']}: {order['item']}, ${order['price']:.2f}, "
                f"status={order['status']}, delivered {order['days_ago']} days ago, "
                f"customer={order['customer']}"
            )
    return (
        f"No order with ID {order_id!r}. Order IDs are 4 digits starting with 10 "
        f"(e.g. 1042). Ask the customer to check their confirmation email."
    )


@tool
def search_tickets(query: str) -> str:
    """Search past support tickets for a keyword.

    Use this to find whether a customer has written in before about the same problem.
    """
    hits = [t for t in TICKETS if query.lower() in t["text"].lower()]
    if not hits:
        return f"No tickets matching {query!r}. Try a shorter keyword."
    return "\n".join(f"{t['id']} (order {t['order_id']}): {t['text']}" for t in hits)


agent = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, search_tickets],
    system_prompt=(
        "You are a customer support agent for an office furniture retailer.\n"
        "Look up the order before answering questions about it.\n"
        "Use the refund-decision skill whenever a remedy is being considered."
    ),
    memory=["/memory/AGENTS.md"],
    skills=["/skills/"],
    backend=CompositeBackend(
        # Anything unrouted is scratch: thread-scoped, gone when the thread is.
        default=StateBackend(),
        routes={
            # Read-only content baked into the image. Scoped per directory, because a
            # route strips its prefix before the backend sees the path.
            "/memory/": FilesystemBackend(root_dir=str(ROOT / "memory")),
            "/skills/": FilesystemBackend(root_dir=str(ROOT / "skills")),
            # In a deployment this resolves to the platform's Postgres store, not memory.
            # The namespace comes from the authenticated caller, so one user's memories
            # are invisible to another. server_info is None outside a deployment.
            "/memories/": StoreBackend(namespace=_memories_namespace),
        },
    ),
    # The image is not a scratchpad. Without this the agent can edit its own AGENTS.md
    # and its own skills, and the change survives for the life of the container.
    permissions=[
        FilesystemPermission(operations=["write"], paths=["/memory/*", "/skills/*"], mode="deny"),
    ],
    middleware=[
        ToolRetryMiddleware(max_retries=3, tools=["lookup_order"], on_failure="continue"),
        PIIMiddleware("email", strategy="redact", apply_to_tool_results=True),
        SummarizationMiddleware(model=MODEL, trigger=("tokens", 8000), keep=("messages", 6)),
    ],
)

### The backend layout is different from the notebook's, on purpose

A deployment is a web server, and handing a web server's filesystem to a model is how you get an
agent that can read your `.env`. `FilesystemBackend` says so in its own docstring: web servers and
HTTP APIs are an inappropriate use, prefer `StateBackend`, `StoreBackend`, or a sandbox.

So each path gets the weakest backend that can do its job:

| Path | Backend | Lifetime |
|---|---|---|
| `/memory/`, `/skills/` | `FilesystemBackend`, one per directory | baked into the image, **read-only** |
| `/memories/` | `StoreBackend` | Postgres, survives everything, scoped per caller |
| everything else | `StateBackend` | the thread, and no longer |

Three things in there are easy to get wrong.

**A route strips its own prefix** before the backend sees the path, which is why each one gets
`root_dir=ROOT / "memory"` rather than a single `root_dir=ROOT`. Point both at `ROOT` and
`/memory/AGENTS.md` becomes a lookup for `ROOT/AGENTS.md`, which does not exist.

**Read-only is not the default.** Without the `deny` permission the agent can edit its own
`AGENTS.md` and its own skills — and because that write lands on the container filesystem, it
persists for the life of the replica, is invisible to your other replicas, and disappears on the
next deploy. An agent quietly rewriting its own instructions is a genuinely bad afternoon.

**`StoreBackend` becomes real.** In the notebook it was an `InMemoryStore` that died with the
kernel. Here it is the platform's Postgres store, so `/memories/` actually persists — and the
`namespace` callable is the only thing keeping one caller's memories out of another's.

---

## 3. `langgraph.json`, field by field

This is the manifest. It is small, and almost every deployment problem is a mistake in it.

In [ ]:
%%writefile support_agent/pyproject.toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "support-agent"
version = "0.1.0"
requires-python = ">=3.11"
dependencies = [
    "deepagents~=0.7.6",
    "langchain~=1.3.15",
    "langchain-openai~=1.5.1",
]

# Without this, setuptools scans for packages, finds data/ memory/ skills/,
# and refuses to build rather than guess. Do not remove.
[tool.setuptools]
py-modules = ["agent"]

In [ ]:
%%writefile support_agent/langgraph.json
{
  "python_version": "3.12",
  "dependencies": ["."],
  "graphs": {
    "support": "./agent.py:agent"
  },
  "env": ".env",
  "dockerfile_lines": ["ENV FF_V2_EVENT_STREAMING=false"]
}

Here is the same file with everything you are likely to reach for, annotated. JSON has no
comments, so this is illustrative — the file you just wrote is the real one.

```js
{
  // Interpreter for the image. deepagents needs 3.11+.
  "python_version": "3.12",

  // What goes in the image. "." means "install this directory", so it needs a
  // pyproject.toml or requirements.txt beside it. Package names work too.
  "dependencies": ["."],

  // "name": "./file.py:variable". The name becomes the ASSISTANT ID you call,
  // and the variable must already exist when the module is imported.
  "graphs": {
    "support": "./agent.py:agent"
  },

  // A dotenv path, or a dict of literal values. Real secrets belong in the
  // platform's secret store, not in a file you might commit.
  "env": ".env",

  // The platform store — this is what makes StoreBackend durable. `index` turns on
  // semantic search over what the agent wrote; `ttl` expires old entries.
  "store": {
    "index": { "embed": "openai:text-embedding-3-small", "dims": 1536, "fields": ["$"] },
    "ttl": { "refresh_on_read": true, "default_ttl": 43200 }
  },

  // Thread persistence. A ttl keeps abandoned conversations from accumulating.
  "checkpointer": {
    "ttl": { "strategy": "delete", "sweep_interval_minutes": 60, "default_ttl": 43200 }
  },

  // A module that authenticates requests and can scope resources per user.
  // Without it, the platform authenticates callers by their LangSmith API key.
  "auth": { "path": "./auth.py:auth" },

  // The server's own HTTP surface: custom routes, and switches that turn
  // features off (`disable_mcp`, `disable_ui`, `disable_runs`, ...).
  "http": { "app": "./app.py:app" },

  // Appended to the generated Dockerfile: system packages, and any
  // environment variable you want baked into the image.
  "dockerfile_lines": [
    "RUN apt-get update && apt-get install -y libpq-dev",
    "ENV FF_V2_EVENT_STREAMING=false"
  ]
}
```

### Why that `ENV` line is there

On `langgraph-api` 0.12.7 the v2 event-streaming transport loses the **arguments** of a tool call.
The name and id survive aggregation, the arguments are dropped, and every tool then arrives with
`{}` and fails its own schema. Studio sets the v2 flag on runs it creates, so a deployment driven
from Studio hits this while the same graph called over the API is fine.

Turning the v2 routes off avoids it, and clients handle their absence already — it is the same
state as an older server. Two knobs claim to do that, and only one of them works:

| Knob | Works on 0.12.7? |
|---|---|
| `"http": {"disable_event_streaming": true}` | **No.** The router reads the key, but `HttpConfig` never declares it, so the config parser drops it on the way in. |
| `FF_V2_EVENT_STREAMING=false` | Yes. Read straight from the environment, with no schema in the way. |

The manifest field is the one you would reach for, and it validates, and it is silently discarded —
so the environment variable goes in `dockerfile_lines`, which keeps it version-controlled with the
code instead of set by hand on one deployment.

Delete the line once the upstream fix ships.

Almost every deployment problem is a mistake in this file, which is why the next cell exists.

---

## 4. Validate before you build

`langgraph validate` checks the manifest and imports every graph it names. It is the cheapest way
to find a typo — seconds, versus minutes for a failed image build.

In [ ]:
!cd support_agent && langgraph validate

If that says `1 graph found`, the manifest is right and `agent.py` imports cleanly. A broken
`graphs` path, a missing dependency, or a syntax error in the agent all fail here.

---

## 5. Deploying it

> **Note.** Workshop accounts do not have permission to create deployments, so the commands in
> this section are here to read rather than run. A shared deployment already exists — you will
> find it and call it in the next section.

The whole thing is four commands:

```bash
cd support_agent

# Build and deploy. --name is what appears in LangSmith Deployments.
langgraph deploy --name support-agent

# Watch it come up
langgraph deploy list
langgraph deploy logs --name support-agent

# Ship a change: same command again creates a new revision
langgraph deploy --name support-agent
langgraph deploy revisions list --name support-agent
```

The platform builds the image from `langgraph.json`, provisions Postgres for the checkpointer and
store, and gives you an HTTPS endpoint. What you get for free is the part worth noticing: threads,
the store, interrupts that survive a restart, and streaming — all the things lessons 02, 06, and 09
showed you, now durable and behind an API.

### Running it locally instead

You can still exercise the exact same manifest without deploying. `langgraph dev` starts the same
server on your machine, reading the same `langgraph.json`:

```bash
cd support_agent && langgraph dev
```

That is what `start_studio()` has been doing for you since lesson 01 — it wrote a temporary
manifest and ran this server. Now you own the manifest.

---

## 6. Find the deployment

You cannot create a deployment, but you can list the ones your workspace has. That is how you get
the URL — no need for anyone to read it out.

In [ ]:
!langgraph deploy list --name-contains support-agent

Copy the URL from that output into the cell below. If the list comes back empty, the shared
deployment is not up yet — ask before continuing.

---

## 7. Calling it as a `RemoteGraph`

A deployment is an HTTP service, but you do not have to treat it like one. `RemoteGraph` gives you
the same interface as a local graph — `.invoke()`, `.stream()`, threads, state — against the
deployed one.

In [ ]:
from langgraph.pregel.remote import RemoteGraph

# From the list above. "support" is the key from "graphs" in langgraph.json — the assistant id.
DEPLOYMENT_URL = "https://<paste-from-the-list-above>"

support = RemoteGraph("support", url=DEPLOYMENT_URL, api_key=key)
support

In [ ]:
def last_text(result: dict) -> str:
    """The final reply. A deployment returns JSON, so messages are dicts."""
    content = result["messages"][-1].get("content") or ""
    if isinstance(content, list):
        return " ".join(part.get("text", "") for part in content if isinstance(part, dict))
    return str(content)


result = support.invoke({"messages": [{"role": "user", "content":
    "Ticket T-6: the laptop stand on order 1047 wobbles. What can we offer?"
}]})

print(last_text(result))

Order 1047 was delivered 62 days ago, so the skill's policy table allows a **repair only**. If you
get an offer of a refund, you have found a bug — and in the next lesson you will turn it into a
test case rather than a note to yourself.

Three things worth noticing about what just happened:

- **The agent ran somewhere else.** Nothing was installed in this kernel to make that call. Your
  notebook is a client.
- **It is still traced.** Open `lcw-10-deploy` in LangSmith and the run is there, with the same
  tool calls and the same skill load you saw in Studio.
- **The messages came back as dicts.** A local graph hands you `AIMessage` objects; a deployment
  hands you JSON over HTTP. So it is `message["content"]` and `message["tool_calls"]`, never
  `message.text` — which is why the helper above exists, and why it has to cope with `content`
  being either a string or a list of blocks.

In [ ]:
# Threads work the same as they did locally — the platform is providing the checkpointer.
config = {"configurable": {"thread_id": "deploy-demo-1"}}

support.invoke({"messages": [{"role": "user", "content": "My name is Avery."}]}, config=config)
second = support.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config=config)

print(last_text(second))

That memory came from Postgres on the deployment, not from anything in this notebook. Restart this
kernel and the thread is still there.

---

## 📌 Key takeaways

- A deployment is a **directory**, not a notebook cell: the platform builds an image from it.
- `langgraph.json` is the whole manifest — `dependencies`, `graphs`, `env`, and a handful of optional fields for the store, checkpointer, and auth.
- `graphs` keys become **assistant ids**; the variable they point at must exist at import time.
- `langgraph validate` imports every graph and catches manifest mistakes in seconds.
- `langgraph dev` runs the same manifest locally — it is what `start_studio()` was doing all along.
- In a deployment, `StoreBackend` stops being ephemeral and becomes Postgres, which is why the `namespace` callable matters.
- A deployment is a web server, so do not hand it the host filesystem — `FilesystemBackend` says as much in its own docstring.
- Give each path the weakest backend that can do its job, and make image content **read-only** with a `deny` permission.
- A `CompositeBackend` route strips its own prefix, so each route needs its own `root_dir`.
- `RemoteGraph` gives a deployed agent the same interface as a local one, tracing included.
- `langgraph deploy list --name-contains` finds a deployment without needing anyone to read out a URL.
- `rt.server_info.user` is the authenticated caller, so a store namespace can isolate users without trusting anything the caller sends.

---

## ➡️ Next

**[11 · Datasets from real traffic](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/11_datasets.ipynb)**

You have a deployed agent and you have just seen it answer something questionable. Next: turning
its real traffic into a dataset, so that "questionable" becomes a test you can re-run.